# RailSight: detector training on Kaggle

Trains one RailSight obstacle detector on the merged railway-obstacle dataset, with the same
settings as `scripts/train_det.py` on the laptop, and packs the results into one zip to download.

**Before running** (right-hand panel → *Session options*):
- Accelerator: **GPU T4 x2**
- Internet: **On** (needed to install Ultralytics and download pretrained weights)
- Input: the private dataset **railsight-det** is attached

Run it with **Save Version → Save & Run All (Commit)** so it keeps going after you close the tab.
Change `MODEL` below and save a second version to train the other detector.

In [ ]:
# ---- choose the run -------------------------------------------------------------------------
MODEL = "yolo11m.pt"      # "yolo11m.pt" or "rtdetr-l.pt"
NAME = "yolo11m_rail"     # "yolo11m_rail" or "rtdetr_l_rail" (must match the laptop's run names)
BATCH = 32                # total over both GPUs: 32 for yolo11m, 16 for rtdetr-l
HOURS = 10.5              # hard time budget; Kaggle stops sessions at 12 h

In [ ]:
!pip install -q ultralytics==8.4.174
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Copy the read-only input to local disk so Ultralytics can write its label cache next to it
import glob
import os
import shutil

src = os.path.dirname(glob.glob("/kaggle/input/**/railsight-det/data.yaml", recursive=True)[0])
DATA = "/tmp/railsight-det"
if not os.path.exists(DATA):
    shutil.copytree(src, DATA)
with open(f"{DATA}/data.yaml") as f:
    yaml_text = f.read()
with open(f"{DATA}/data.yaml", "w") as f:
    f.write("path: " + DATA + chr(10) + yaml_text)
for split in ("train", "val", "test"):
    print(split, len(os.listdir(f"{DATA}/{split}/images")), "images")

In [ ]:
# Multi-GPU training is launched from a script: Ultralytics' DDP is reliable that way
import torch

devices = ",".join(str(i) for i in range(torch.cuda.device_count())) or "cpu"
script = f'''
from ultralytics import RTDETR, YOLO
cls = RTDETR if "rtdetr" in "{MODEL}" else YOLO
cls("{MODEL}").train(
    data="{DATA}/data.yaml", epochs=100, imgsz=640, batch={BATCH}, workers=4,
    device="{devices}", project="/kaggle/working/runs/det", name="{NAME}", exist_ok=True,
    patience=30, cos_lr=True, close_mosaic=10, seed=0, plots=True, time={HOURS},
)
'''
with open("train.py", "w") as f:
    f.write(script)
print(script)

In [ ]:
!python train.py

In [ ]:
# Held-out test split, single GPU
from ultralytics import RTDETR, YOLO

cls = RTDETR if "rtdetr" in MODEL else YOLO
best = f"/kaggle/working/runs/det/{NAME}/weights/best.pt"
m = cls(best).val(data=f"{DATA}/data.yaml", split="test", device=0, plots=False,
                  project="/kaggle/working/runs/det_eval", name=NAME, exist_ok=True)
print(f"test mAP50={m.box.map50:.3f}  mAP50-95={m.box.map:.3f}")

In [ ]:
# One zip to download: Output tab → <NAME>.zip. Unzip into the laptop's RailSight/data/incoming/
shutil.make_archive(f"/kaggle/working/{NAME}", "zip", "/kaggle/working/runs/det", NAME)
shutil.rmtree("/kaggle/working/runs")
os.remove("train.py")
print(os.listdir("/kaggle/working"))